# Lab 6: nnU-Net v2 Self-Configuring Segmentation on SageMaker

This notebook demonstrates how to train a medical image segmentation model on Amazon SageMaker using nnU-Net v2. nnU-Net automatically determines optimal preprocessing, architecture, and training hyperparameters.

## What You'll Learn
- Running nnU-Net v2 on SageMaker with a custom Docker image
- Self-configuring preprocessing, training, and evaluation
- Comparing nnU-Net with manually configured MONAI models (Lab 1)

## Prerequisites
- Medical imaging data in S3 in nnU-Net format (`imagesTr/`, `labelsTr/`, `dataset.json`)
- SageMaker execution role with S3 access

## Step 1: Setup and Imports

In [ ]:
import os
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.local import LocalSession
from sagemaker import get_execution_role
import boto3

sagemaker_session = sagemaker.Session(boto3.Session(region_name='us-east-1'))
# Dedicated SageMaker execution role — use this when running outside a SageMaker notebook
# role = get_execution_role()
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
print(f"SageMaker role: {role}")
print(f"Region: {region}")
print(f"Bucket: {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ubuntu/.config/sagemaker/config.yaml
SageMaker role: arn:aws:iam::575108919340:role/service-role/AmazonSageMaker-ExecutionRole-20240907T181142
Region: us-east-1
Bucket: sagemaker-us-east-1-575108919340


## Step 1b: Create SageMaker Execution Role (one-time setup)

Run this cell once to create a dedicated SageMaker execution role with the same permissions as the default role. Skip if the role already exists.

In [4]:
import json

iam = boto3.client('iam')
role_name = "AmazonSageMaker-ExecutionRole-nnunet"

# Managed policies — mirrors AmazonSageMaker-ExecutionRole-20240907T181142
managed_policies = [
    "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
    "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    "arn:aws:iam::aws:policy/AmazonRekognitionFullAccess",
    "arn:aws:iam::aws:policy/ComprehendFullAccess",
    "arn:aws:iam::aws:policy/ComprehendMedicalFullAccess",
    "arn:aws:iam::aws:policy/AmazonElasticContainerRegistryPublicFullAccess",
    "arn:aws:iam::aws:policy/AmazonSageMakerCanvasFullAccess",
    "arn:aws:iam::aws:policy/AmazonSageMakerCanvasAIServicesAccess",
    "arn:aws:iam::aws:policy/AmazonSageMakerCanvasSMDataScienceAssistantAccess",
    "arn:aws:iam::aws:policy/AmazonSageMakerCanvasDataPrepFullAccess",
]

# Inline policy for MLflow
mlflow_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["sagemaker-mlflow:*"],
        "Resource": "*"
    }]
}

# Trust policy — allows SageMaker service to assume this role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    existing = iam.get_role(RoleName=role_name)
    role_arn = existing['Role']['Arn']
    print(f"Role already exists: {role_arn}")
except iam.exceptions.NoSuchEntityException:
    # Create role
    response = iam.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="SageMaker execution role for nnU-Net training"
    )
    role_arn = response['Role']['Arn']
    print(f"Created role: {role_arn}")

    # Attach managed policies
    for policy_arn in managed_policies:
        iam.attach_role_policy(RoleName=role_name, PolicyArn=policy_arn)
        print(f"  Attached: {policy_arn.split('/')[-1]}")

    # Add inline MLflow policy
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName="mlflow-policy",
        PolicyDocument=json.dumps(mlflow_policy)
    )
    print("  Added inline policy: mlflow-policy")

# Use this role for all estimators
role = role_arn
print(f"\nUsing role: {role}")

Role already exists: arn:aws:iam::575108919340:role/AmazonSageMaker-ExecutionRole-nnunet

Using role: arn:aws:iam::575108919340:role/AmazonSageMaker-ExecutionRole-nnunet


In [5]:
# Remote S3 paths — used for the real SageMaker run in Step 5
data_bucket = "public-datasets-imaging-us-east-1"
data_path = f"s3://{data_bucket}/nnUNet/"
output_path = f"s3://{bucket}/nnunet-segmentation/output"

# Local paths — used for local mode test in Step 4b
# Point local_data_path at a small dataset in nnU-Net format: imagesTr/, labelsTr/, dataset.json
local_data_path = os.path.abspath("../data/sample")
local_output_path = os.path.abspath("../output/local")
os.makedirs(local_output_path, exist_ok=True)

print(f"Remote training data : {data_path}")
print(f"Remote output path   : {output_path}")
print(f"Local training data  : {local_data_path}")
print(f"Local output path    : {local_output_path}")

Remote training data : s3://public-datasets-imaging-us-east-1/nnUNet/
Remote output path   : s3://sagemaker-us-east-1-575108919340/nnunet-segmentation/output
Local training data  : /home/ubuntu/tools/pr-12-review/medical-image-segmentation/data/sample
Local output path    : /home/ubuntu/tools/pr-12-review/medical-image-segmentation/output/local


## Step 3: Build and Push Docker Image

nnU-Net uses a separate Dockerfile since it has different dependencies than MONAI.

> **Local mode** only needs the local image tag — no ECR push required until Step 5.

In [6]:
account_id = boto3.client('sts').get_caller_identity()['Account']
image_name = "nnunet-segmentation"
local_image = f"{image_name}:latest"
ecr_repo = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"

print(f"Local image    : {local_image}")
print(f"ECR repository : {ecr_repo}")
print()
print("Build the image (required for both local mode and SageMaker):")
print(f"  cd ../code")
print(f"  docker build -f docker/Dockerfile.nnunet -t {local_image} .")
print()
print("Push to ECR (only needed for Step 5 — SageMaker remote run):")
print(f"  aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com")
print(f"  aws ecr create-repository --repository-name {image_name} --region {region} || true")
print(f"  docker tag {local_image} {ecr_repo}")
print(f"  docker push {ecr_repo}")

Local image    : nnunet-segmentation:latest
ECR repository : 575108919340.dkr.ecr.us-east-1.amazonaws.com/nnunet-segmentation:latest

Build the image (required for both local mode and SageMaker):
  cd ../code
  docker build -f docker/Dockerfile.nnunet -t nnunet-segmentation:latest .

Push to ECR (only needed for Step 5 — SageMaker remote run):
  aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin 575108919340.dkr.ecr.us-east-1.amazonaws.com
  aws ecr create-repository --repository-name nnunet-segmentation --region us-east-1 || true
  docker tag nnunet-segmentation:latest 575108919340.dkr.ecr.us-east-1.amazonaws.com/nnunet-segmentation:latest
  docker push 575108919340.dkr.ecr.us-east-1.amazonaws.com/nnunet-segmentation:latest


## Step 4a: Create SageMaker Estimator (remote)

This estimator targets a real SageMaker instance. It is used in Step 5.

In [7]:
estimator = Estimator(
    image_uri=ecr_repo,
    entry_point="nnunet_pipeline.py",
    source_dir="../code/training/nnunet",
    role=role,
    instance_count=1,
    instance_type="ml.g5.xlarge",
    hyperparameters={
        "stages": "preprocess,train,evaluate",
        "num_epochs": 5
    },
    output_path=output_path,
    base_job_name="nnunet-pipeline",
    keep_alive_period_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)
print("Remote estimator created successfully!")

Remote estimator created successfully!


## Step 4b: Local Mode Test

Runs the exact same container locally via Docker before submitting to SageMaker.
Validates the container entrypoint, script paths, and data loading without incurring cloud compute costs.

**Requirements:**
- Docker running locally
- `sagemaker[local]` installed: `pip install 'sagemaker[local]'`
- Image built locally (Step 3 build command — no push needed)
- Sample data at `../data/sample/` in nnU-Net format (`imagesTr/`, `labelsTr/`, `dataset.json`)

In [7]:
# LocalSession must be used with instance_type="local" or "local_gpu"
local_session = LocalSession()
local_session.config = {
    'local': {
        'local_code': True,       # mount source_dir directly — no repackaging
        'container_config': {
            'shm_size': '8g',     # nnU-Net data workers need large shared memory
        }
    }
}

local_estimator = Estimator(
    image_uri=local_image,            # local Docker image — no ECR pull
    entry_point="nnunet_pipeline.py",
    source_dir="../code/training/nnunet",
    role=role,
    instance_count=1,
    instance_type="local_gpu",        # change to "local" if no GPU available
    hyperparameters={
        "stages": "preprocess,train,evaluate",
        "num_epochs": 5
    },
    output_path=f"file://{local_output_path}",
    sagemaker_session=local_session,
)
print("Local estimator created!")
print(f"  Image  : {local_image}")
print(f"  Data   : {local_data_path}")
print(f"  Output : {local_output_path}")

INFO:botocore.credentials:Found credentials from IAM Role: ADMINACCESS


Local estimator created!
  Image  : nnunet-segmentation:latest
  Data   : /home/ubuntu/tools/pr-12-review/medical-image-segmentation/data/sample
  Output : /home/ubuntu/tools/pr-12-review/medical-image-segmentation/output/local


In [ ]:
# Kick off local training — uses file:// channel so no S3 access needed
local_estimator.fit(
    {"training": f"file://{local_data_path}"},
    wait=True,
    logs="All",
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: nnunet-segmentation-2026-06-06-07-09-02-921
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found us

time="2026-06-06T07:09:03Z" level=warning msg="/tmp/tmp5im4xjf4/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-06-06T07:09:03Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmp5im4xjf4\".\nSet `external: true` to use an existing network"
 Container vdkir9h0cq-algo-1-bzvij  Creating
 Container vdkir9h0cq-algo-1-bzvij  Created
Attaching to vdkir9h0cq-algo-1-bzvij
vdkir9h0cq-algo-1-bzvij  | 2026-06-06 07:09:05,547 sagemaker-training-toolkit INFO     Provided path: /opt/ml/code is not empty, abandoning unzipping sourcedir.tar.gz
vdkir9h0cq-algo-1-bzvij  | 2026-06-06 07:09:05,570 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
vdkir9h0cq-algo-1-bzvij  | 2026-06-06 07:09:05,576 sagemaker-training-toolkit INFO     instance_groups entry not present in resource_config
vdkir9h0cq-algo-1-bzvij  | 2026-06-06 07:09:05,60

## Step 5: Start Training on SageMaker

Once local mode passes, submit the full job to SageMaker.

In [8]:
estimator.fit({"training": data_path}, wait=True, logs="All")

INFO:sagemaker:Creating training-job with name: nnunet-pipeline-2026-06-06-07-23-07-656


2026-06-06 07:23:08 Starting - Starting the training job
2026-06-06 07:23:08 Pending - Training job waiting for capacity.................................
2026-06-06 07:28:30 Pending - Preparing the instances for training...
2026-06-06 07:29:18 Downloading - Downloading the training image............
2026-06-06 07:30:59 Training - Training image download completed. Training in progress.2026-06-06 07:31:07,648 sagemaker-training-toolkit INFO     Provided path: /opt/ml/code  is empty, unzipping
2026-06-06 07:31:08,007 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-06 07:31:08,035 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-06 07:31:08,062 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-06 07:31:08,071 sagemaker-training-toolkit INFO     Invoking user script
Training Env:
{
    "additional_framework_parameters": {},
    "channel_input_dirs"

## Step 6: View Training Results

In [9]:
training_job_name = estimator.latest_training_job.name
model_data = estimator.model_data
print(f"Training job    : {training_job_name}")
print(f"Model artifacts : {model_data}")

Training job    : nnunet-pipeline-2026-06-06-07-23-07-656
Model artifacts : s3://sagemaker-us-east-1-575108919340/nnunet-segmentation/output/nnunet-pipeline-2026-06-06-07-23-07-656/output/model.tar.gz
